# VQ-GAN and ViT VQ-GAN: Discrete Image Tokenizers

## Theory, mathematics, and implementation context

A vector-quantized autoencoder replaces each encoder feature with its nearest codebook vector: $k^*=\arg\min_k\|z_e-e_k\|^2$, $z_q=e_{k^*}$. Its objective includes reconstruction, codebook, and commitment losses. Nearest-neighbor lookup is non-differentiable, so a straight-through estimator uses $z_q^{ST}=z_e+\mathrm{sg}(z_q-z_e)$ to pass decoder gradients to the encoder. VQ-GAN adds adversarial and often perceptual losses so reconstructions preserve perceptual detail, yielding compact discrete tokens for a separate prior such as a Transformer.

“ViT VQ-GAN” is not a single canonical architecture: it commonly means a VQ tokenizer with Vision Transformer components in its encoder/decoder or discriminator. This notebook uses a ViT patch encoder, vector quantizer, convolutional decoder, and optional patch discriminator. It is a small 32px CIFAR-10 teaching tokenizer, not a production tokenizer or complete token prior.

## Case study: compact CIFAR-10 tokenizer

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
torch.manual_seed(7)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf=transforms.Compose([transforms.Resize(32),transforms.ToTensor(),transforms.Normalize((.5,)*3,(.5,)*3)])
ds=datasets.CIFAR10("./data",train=True,download=True,transform=tf)
loader=DataLoader(Subset(ds,range(4096)),batch_size=64,shuffle=True,num_workers=0,drop_last=True)
print(DEVICE,len(loader.dataset))

## Patch embedding and ViT encoder

Non-overlapping patches become a token sequence with positional embeddings. Self-attention gives each patch global context.

In [ ]:
class ViTEncoder(nn.Module):
    def __init__(self,image=32,patch=4,dim=128,depth=2,heads=4):
        super().__init__(); self.grid=image//patch; n=self.grid**2; self.patch=nn.Conv2d(3,dim,patch,stride=patch); self.pos=nn.Parameter(torch.randn(1,n,dim)*.02)
        layer=nn.TransformerEncoderLayer(dim,heads,dim*4,batch_first=True,norm_first=True); self.transformer=nn.TransformerEncoder(layer,depth); self.proj=nn.Linear(dim,128)
    def forward(self,x):
        t=self.patch(x).flatten(2).transpose(1,2)+self.pos; t=self.proj(self.transformer(t)); b,n,c=t.shape
        return t.transpose(1,2).reshape(b,c,self.grid,self.grid)
enc=ViTEncoder().to(DEVICE); print(enc(torch.randn(2,3,32,32,device=DEVICE)).shape)


## Vector quantizer and straight-through estimator

Squared distance uses $\|z-e\|^2=\|z\|^2+\|e\|^2-2z^Te$. Perplexity summarizes code usage; low perplexity may indicate codebook collapse.

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self,k=512,d=128,beta=.25):
        super().__init__(); self.embedding=nn.Embedding(k,d); self.embedding.weight.data.uniform_(-1/k,1/k); self.k=k; self.beta=beta
    def forward(self,z):
        b,c,h,w=z.shape; flat=z.permute(0,2,3,1).reshape(-1,c); e=self.embedding.weight
        dist=flat.square().sum(1,keepdim=True)+e.square().sum(1)-2*flat@e.t(); ids=dist.argmin(1)
        q=self.embedding(ids).view(b,h,w,c).permute(0,3,1,2).contiguous()
        loss=F.mse_loss(q,z.detach())+self.beta*F.mse_loss(q.detach(),z)
        q_st=z+(q-z).detach(); usage=F.one_hot(ids,self.k).float().mean(0); ppl=torch.exp(-(usage*torch.log(usage+1e-10)).sum())
        return q_st,loss,ids.view(b,h,w),ppl
q,vq,ids,ppl=VectorQuantizer().to(DEVICE)(enc(torch.randn(2,3,32,32,device=DEVICE))); print(q.shape,ids.shape,float(ppl))

## Decoder and hybrid ViT VQ model

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(128,128,3,padding=1),nn.GELU(),nn.Upsample(scale_factor=2,mode="nearest"),nn.Conv2d(128,64,3,padding=1),nn.GELU(),nn.Upsample(scale_factor=2,mode="nearest"),nn.Conv2d(64,32,3,padding=1),nn.GELU(),nn.Conv2d(32,3,3,padding=1),nn.Tanh())
    def forward(self,z): return self.net(z)
class ViTVQ(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=ViTEncoder(); self.quantizer=VectorQuantizer(k=256); self.decoder=Decoder()
    def forward(self,x):
        z=self.encoder(x); q,loss,ids,ppl=self.quantizer(z); return self.decoder(q),loss,ids,ppl
model=ViTVQ().to(DEVICE); print(model(torch.randn(2,3,32,32,device=DEVICE))[0].shape)


## Reconstruction-first optimization

The trainable parameters are collected here. The full loop below trains the tokenizer before displaying reconstructions.


In [ ]:
opt=torch.optim.Adam(model.parameters(),lr=2e-4)
x,_=next(iter(loader)); x=x.to(DEVICE); recon,vq_loss,ids,ppl=model(x); recon_loss=F.l1_loss(recon,x); loss=recon_loss+vq_loss
opt.zero_grad(); loss.backward(); opt.step(); print({"total":float(loss),"l1":float(recon_loss),"vq":float(vq_loss),"perplexity":float(ppl)})

## Patch discriminator and adversarial objective

VQ-GAN combines pixel/perceptual reconstruction, codebook loss, and adaptively weighted adversarial loss. This self-contained demonstration uses L1 plus hinge loss and omits pretrained perceptual features.

In [ ]:
from torch.nn.utils import spectral_norm
class PatchD(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(spectral_norm(nn.Conv2d(3,32,4,2,1)),nn.LeakyReLU(.2),spectral_norm(nn.Conv2d(32,64,4,2,1)),nn.LeakyReLU(.2),spectral_norm(nn.Conv2d(64,1,3,padding=1)))
    def forward(self,x): return self.net(x)
disc=PatchD().to(DEVICE); print("Patch discriminator ready")


## Train the tokenizer and inspect reconstructions

The earlier version performed only one update. This compact ViT tokenizer and patch discriminator now train for 40 epochs on 4,096 CIFAR-10 images. The adversarial term ramps in gradually so the tokenizer first learns to preserve image content. On CPU, training may take several minutes.


In [ ]:
import time
EPOCHS=40
ADVERSARIAL_WEIGHT=0.05
tok_opt=torch.optim.Adam(model.parameters(),lr=2e-4)
d_opt=torch.optim.Adam(disc.parameters(),lr=2e-4)
g_history=[]; d_history=[]
for epoch in range(EPOCHS):
    model.train(); disc.train(); g_running=0.0; d_running=0.0
    adv_weight=ADVERSARIAL_WEIGHT*min(1.0,(epoch+1)/5)
    for x,_ in loader:
        x=x.to(DEVICE)
        recon,vq_loss,_,_=model(x)
        d_loss=F.relu(1-disc(x)).mean()+F.relu(1+disc(recon.detach())).mean()
        d_opt.zero_grad(set_to_none=True); d_loss.backward(); d_opt.step()
        for parameter in disc.parameters(): parameter.requires_grad_(False)
        recon,vq_loss,_,_=model(x)
        recon_loss=F.l1_loss(recon,x)
        g_loss=recon_loss+vq_loss+adv_weight*(-disc(recon).mean())
        tok_opt.zero_grad(set_to_none=True); g_loss.backward(); tok_opt.step()
        for parameter in disc.parameters(): parameter.requires_grad_(True)
        g_running+=g_loss.item(); d_running+=d_loss.item()
    g_history.append(g_running/len(loader)); d_history.append(d_running/len(loader))
    print(f"epoch {epoch+1:02d}/{EPOCHS} | tokenizer={g_history[-1]:.3f} | D={d_history[-1]:.3f} | adv={adv_weight:.3f}")
plt.figure(figsize=(9,3)); plt.plot(g_history,label="Tokenizer"); plt.plot(d_history,label="Discriminator"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=.25)


## Reconstruction and codebook diagnostics

The left row contains real images; the right row contains reconstructions from the trained tokenizer. This checks that the model learned image structure. Unconditional sampling would require an additional prior trained over code sequences, as described above.


In [ ]:
model.eval()
with torch.no_grad():
    real,_=next(iter(loader)); real=real[:8].to(DEVICE)
    recon,_,indices,perplexity=model(real)
comparison=torch.cat([real.cpu(),recon.cpu()])
plt.figure(figsize=(10,4)); plt.axis("off"); plt.imshow(utils.make_grid(comparison,nrow=8,normalize=True).permute(1,2,0));
print("codebook perplexity:",float(perplexity),"active codes:",indices.unique().numel())


## Key Takeaways

- VQ maps continuous image features to discrete indices by nearest-neighbor lookup.
- The straight-through estimator routes decoder gradients through quantization.
- Codebook, commitment, and reconstruction losses play different roles; perplexity diagnoses usage.
- VQ-GAN adds perceptual/adversarial pressure for realistic reconstructions.
- A separate prior over code sequences is required for unconditional generation.
- ViT VQ-GAN is a family of tokenizer designs, not one standardized architecture.
- This compact CIFAR-10 model teaches mechanics, not state-of-the-art fidelity.